# Macro features for Numerai Signals

This notebook attaches each stock's home-economy policy rate, inflation, unemployment and GDP growth to the Numerai Signals universe, as they stood before each Signals date. Data comes from [FXMacroData](https://fxmacrodata.com/?utm_source=github&utm_medium=notebook&utm_campaign=numerai-signals&utm_content=notebook-intro).

US-listed stocks work without an API key. Set `FXMACRODATA_API_KEY` to cover the other 20 currencies.

In [ ]:
# pip install numerapi pyarrow
# pip install .   (from the numerai-signals directory)
import os
import pandas as pd
from numerapi import SignalsAPI
from fxmacrodata_numerai import (
    CONFIRMED_ONLY, MacroHistoryClient, example_signal, macro_panel, map_tickers, ticker_features,
)

## 1. Download the Signals universe

Only the ticker and date columns are needed.

In [ ]:
napi = SignalsAPI()
napi.download_dataset("signals/v3.1/validation.parquet", "validation.parquet")
napi.download_dataset("signals/v3.1/live.parquet", "live.parquet")

validation = pd.read_parquet("validation.parquet", columns=["numerai_ticker", "date"])
live = pd.read_parquet("live.parquet")
live = live[["numerai_ticker"] + (["date"] if "date" in live else [])]
validation.shape, live.shape

## 2. Which economies does the universe cover?

In [ ]:
latest = validation[validation["date"] == validation["date"].max()]
coverage = map_tickers(latest["numerai_ticker"]).fillna("not covered").value_counts()
coverage.to_frame("stocks")

## 3. Download first-release macro histories

`revisions=first` keeps each value as originally published, with its publication timestamp, so later revisions never reach earlier dates.

In [ ]:
client = MacroHistoryClient()
currencies = sorted(map_tickers(validation["numerai_ticker"]).dropna().unique())
if not client.api_key:
    currencies = ["usd"]

rows = client.histories(currencies, ["policy_rate", "inflation", "unemployment", "gdp"], start_date="2017-01-01")
rows.groupby(["currency", "publication_time_status"]).size().unstack(fill_value=0)

## 4. Build point-in-time features

Each date's cutoff is 00:00 UTC, so a figure released on the feature date itself is not used. For the strictest backtest pass `statuses=CONFIRMED_ONLY`; it keeps only publication times confirmed against the publisher, which shortens history.

In [ ]:
panel = macro_panel(rows, validation["date"])
features = ticker_features(validation, panel)
features.filter(regex="^(numerai_ticker|date|currency|real_policy_rate|inflation_level|gdp_pct_chg_1)$").tail()

## 5. Combine with stock features

Macro values are shared by every stock in a country, so on their own they rank countries. Interactions with stock characteristics let a model treat, say, high-beta stocks differently when real rates are rising.

In [ ]:
stock = pd.read_parquet("validation.parquet", columns=["numerai_ticker", "date", "feature_beta_factor"])
merged = stock.merge(features, on=["numerai_ticker", "date"], how="left")
merged["beta_x_real_rate"] = merged["feature_beta_factor"] * (merged["real_policy_rate_rank"] - 0.5)
merged[["feature_beta_factor", "real_policy_rate_rank", "beta_x_real_rate"]].describe()

## 6. Example live submission

`example_signal` only demonstrates the submission format; it is not a tested strategy.

In [ ]:
live_date = pd.Timestamp.now(tz="UTC").normalize().tz_localize(None)
live_universe = live.assign(date=live.get("date", live_date))
live_panel = macro_panel(rows, live_universe["date"])
live_features = ticker_features(live_universe, live_panel)
submission = example_signal(live_features)
submission = submission[submission["date"] == submission["date"].max()][["numerai_ticker", "signal"]]
submission.to_csv("fxmacrodata_macro_signal.csv", index=False)
submission.describe()

Full indicator list: [FXMacroData API reference](https://fxmacrodata.com/documentation/reference?utm_source=github&utm_medium=notebook&utm_campaign=numerai-signals&utm_content=notebook-reference). Not affiliated with or endorsed by Numerai.